In [ ]:
import os
import shutil
import csv
import json
import glob

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

## Basic Utilities ====================================

In [ ]:
def copy_file(source_path, destination_dir):
    """
    Copy file from sourch path to target directory. 
    Use for copying disparate datasets to centralized directory.
    """
    destination_dir = os.path.expanduser(destination_dir)
    os.makedirs(destination_dir, exist_ok=True)
    filename = os.path.basename(source_path)
    destination_path = os.path.join(destination_dir, filename)

    if os.path.exists(destination_path):
        print(f"{filename} already exists in {destination_dir}")
        return False
    try:
        shutil.copy2(source_path, destination_dir)
        return True
    except Exception:
        print(f"Error copying {filename}")
        return False

def move_file(source_path, destination_dir):
    """
    Move file from sourch path to target directory. 
    Use for moving disparate datasets to centralized directory.
    """
    destination_dir = os.path.expanduser(destination_dir)
    os.makedirs(destination_dir, exist_ok=True)
    filename = os.path.basename(source_path)
    destination_path = os.path.join(destination_dir, filename)

    if os.path.exists(destination_path):
        print(f"{filename} already exists in {destination_dir}")
        return False
    try:
        shutil.move(source_path, destination_dir)
        return True
    except Exception:
        print(f"Error moving {filename}")
        return False

def to_snake_case(name):
    """
    Many datasets and folders containing spaces and special characters.
    Snake case for easier handling.
    """
    name = name.replace('.', '_').replace('-', '_')
    name = ''.join(c.lower() if c.isalnum() or c == '_' else '_' for c in name)
    while '__' in name:
        name = name.replace('__', '_')
    return name.strip('_')

## Census Dataset Utilities ====================================

In [ ]:
def map_census_aliases(file_path, column_mapping_dict):
    """
    Census datasets are double headered. 2 part function to handle this:
    1. Map column aliases to column codes and add dict mapping to corresponding 
    dataset dict entry.
    2. Create a multiindex pandas dataframe using codes and aliases 

    Returns:
    Multiindex pandas dataframe using codes and aliases
    """
    file_path = os.path.expanduser(file_path)

    # Map aliases to column codes using csv module to properly handle quoted fields
    with open(file_path, 'r', newline='', encoding='utf-8-sig') as f:
        csv_reader = csv.reader(f)
        codes = next(csv_reader)  # First row contains codes
        names = next(csv_reader)  # Second row contains names

        # Clean names
        names = [name.strip('\'"') for name in names]
        names = [' '.join(name.split()) for name in names]

        # Update provided dictionary directly
        column_mapping_dict.update(dict(zip(names, codes)))

    # Read data using column aliases
    df = pd.read_csv(file_path, skiprows=[0], encoding='utf-8-sig') 

    # Create MultiIndex columns with names as primary level
    df.columns = pd.MultiIndex.from_tuples(
        list(zip(codes, names)),
        names=['Code', 'Alias']
    )

    return df

def census_drop_cols(df, cols_to_drop):
    """
    Drop columns from multiindex dataframes based on alias header.
    Drop listed estimate columns and corresponding margin of error columns.
    Drop Empty columns.

    Returns:
    Multiindex pandas dataframe with estimate and margin of error columns dropped.
    """
    column_names = df.columns.names

    aliases = df.columns.get_level_values('Alias')
    codes = df.columns.get_level_values('Code')

    # Create a mapping of cleaned aliases to their original columns
    alias_to_col = {}
    for col, alias in zip(df.columns, aliases):
        # Clean alias: handle NaN and strip quotes/whitespace/colons
        clean_alias = "" if pd.isna(alias) else str(alias).strip('"').strip().rstrip(':')
        alias_to_col[clean_alias] = col

    # Initialize columns to drop
    columns_to_drop = set()

    # Drop specified columns and their margin of error pairs
    for remove_col in cols_to_drop:
        # Clean the column name we're looking for
        clean_remove_col = "" if pd.isna(remove_col) else str(remove_col).strip('"').strip().rstrip(':')

        # Look for exact matches in cleaned aliases
        if clean_remove_col in alias_to_col:
            columns_to_drop.add(alias_to_col[clean_remove_col])

            # If this is an estimate column, find and drop corresponding margin of error
            if clean_remove_col.startswith('Estimate!!'):
                margin_col = 'Margin of Error!!' + clean_remove_col[len('Estimate!!'):]
                if margin_col in alias_to_col:
                    columns_to_drop.add(alias_to_col[margin_col])

    # Drop only columns that have both missing/empty headers AND no values
    for col, alias, code in zip(df.columns, aliases, codes):
        # Check for missing or empty headers in either level using consistent cleaning
        clean_alias = "" if pd.isna(alias) else str(alias).strip('"').strip().rstrip(':')
        clean_code = "" if pd.isna(code) else str(code).strip('"').strip().rstrip(':')

        has_empty_header = (clean_alias == '' or clean_code == '')
        has_values = not df[col].isna().all()

        if has_empty_header and not has_values:
            columns_to_drop.add(col)

    # Convert set back to list for dropping
    columns_to_drop = list(columns_to_drop)

    # print(fr"Dropping {len(columns_to_drop)} columns ({len(columns_to_drop)//2} pairs)")
    # print(columns_to_drop)

    cleaned_df = df.drop(columns=columns_to_drop)
    cleaned_df.columns.names = column_names

    return cleaned_df

def export_census_csv(df, output_dir, filename, overwrite=False):
    """
    Export a multiindex dataframe to a single header csv (Aliases, drop Codes header).
    If overwrite, then overwrites existing files with same name, else throw error
    """
    try:
        output_path = os.path.join(os.path.expanduser(output_dir), filename)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # Check if file exists and we're not overwriting
        if os.path.exists(output_path) and not overwrite:
            raise FileExistsError(f"File {filename} already exists and overwrite=False")

        # Get both column headers levels
        aliases = df.columns.get_level_values('Alias')
        cleaned_aliases = [alias.replace('"', '').replace('"', '').strip() 
                           if isinstance(alias, str) else alias for alias in aliases]

        temp_df = df.copy()

        # Write to csv using csv module to properly handle commas in column names
        with open(output_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            # Write only the aliases row
            writer.writerow(cleaned_aliases)

            temp_df.columns = cleaned_aliases
            temp_df.to_csv(f, index=False, header=False)

        print(f"Exported to {output_path}")
        return True

    except Exception as e:
        print(f"Error exporting CSV: {str(e)}")

### Dataset processing pipeline ====================================

In [ ]:
def load_dataset_config(config_path, base_path=""):
    with open(os.path.expanduser(config_path), 'r') as f:
        config = json.load(f)

    if base_path:
        base_path = os.path.expanduser(base_path)
        for dataset in config['datasets']:
            if not os.path.isabs(dataset['original_file_path']):
                dataset['original_file_path'] = os.path.join(base_path, dataset['original_file_path'])

    return config['datasets']

def process_census_dataset(key, alias, original_file_path, file_blocks, central_path_head):
    centralized_file_dir = os.path.join(central_path_head, alias)
    file_blocks[key]['original_file_path'] = original_file_path
    file_blocks[key]['centralized_file_dir'] = centralized_file_dir

    file_name = os.path.basename(original_file_path)
    file_path = os.path.join(centralized_file_dir, file_name)

    copy_file(original_file_path, centralized_file_dir)

    cols_to_drop = file_blocks[key]['cols_to_drop']
    column_mapping_dict = file_blocks[key]['code_to_alias_column_mappings']

    print(f"Processing {alias}")

    alias_df = map_census_aliases(file_path, column_mapping_dict)    
    processed_df = census_drop_cols(alias_df, cols_to_drop)

    return processed_df

def load_and_process_all_datasets(config_path, base_path, file_blocks, central_path_head):
    dataset_configs = load_dataset_config(config_path, base_path)

    processed_datasets = {}

    for config in dataset_configs:
        try:
            df = process_census_dataset(
                key=config['key'],
                alias=config['alias'],
                original_file_path=config['original_file_path'],
                file_blocks=file_blocks,
                central_path_head=central_path_head
            )
            processed_datasets[config['alias']] = df

        except Exception as e:
            print(f"Error processing {config['alias']}: {e}")
        print("-"*30)

    return processed_datasets

In [ ]:
path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/")
notes_wb_path = os.path.join(path_head, "Metrics/Notes/JPL_CCSVI_all_fields.xlsx")

In [ ]:
notes_wb = load_workbook(notes_wb_path)
sheet = notes_wb.active

In [ ]:
file_blocks = {}
current_file_name = None
cols_not_to_drop = ["Geography", "Geographic Area Name"] 

# Iterate through rows to find blocks for each csv file
for row in sheet.iter_rows(min_row=1, max_col=3, values_only=False):
    cell_value = row[0].value
    is_bold = row[0].font.bold if row[0].font else False

    # Detect file block by bold file name
    if is_bold and cell_value:
        original_name = cell_value.strip()
        current_file_name = to_snake_case(original_name)
        file_blocks[current_file_name] = {
            'original_name': original_name,
            'cols_to_drop': [],
            'code_to_alias_column_mappings': {},
            'original_file_path': '',
            'centralized_file_dir': '' 
        }
        continue

    # Check for columns with a "Subfield to keep" value
    if current_file_name and any(col.value for col in row):

        col_name = row[0].value
        subfield_value = row[1].value
        # Track columns with no value in the subfields to keep column
        if col_name and not subfield_value:
            if col_name not in cols_not_to_drop:
                cleaned_col_name = col_name.strip().strip('\'"')
                cleaned_col_name = ' '.join(cleaned_col_name.split())
                file_blocks[current_file_name]['cols_to_drop'].append(cleaned_col_name)

In [ ]:
list(file_blocks.items())

In [ ]:
# exposures_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures")
central_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/JPL")
cleaned_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/cleaned-data")
json_dir_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/census-jsons")

In [ ]:
config_path = "./data/census_datasets_config.json"
base_path = "~/Desktop/Nextcloud/SCOVI Project/Metrics/"

In [ ]:
datasets = load_and_process_all_datasets(config_path, base_path, file_blocks, central_path_head)

In [ ]:
# datasets.keys()

### Age of Structure

##### JPL Notes Cleaning

In [ ]:
age_of_structure_df = datasets['age_of_structure']
age_of_structure_df.head(2)

In [ ]:
cleaned_age_of_structure_df = age_of_structure_df.iloc[:, :2].copy()

cols_to_sum = []

for col in age_of_structure_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        cols_to_sum.append(col)

total_col = ('CALCULATED', 'Total Housing Built Before 1990')
cleaned_age_of_structure_df[total_col] = age_of_structure_df[cols_to_sum].sum(axis=1)

cleaned_age_of_structure_df.head(2)

In [ ]:
export_census_csv(cleaned_age_of_structure_df, cleaned_path_head, "age_of_structure.csv", True)

### Aggregate number of vehicles

In [ ]:
aggregate_vehicles_df = datasets['aggregate_vehicles']
aggregate_vehicles_df.head(2)

In [ ]:
export_census_csv(aggregate_vehicles_df, cleaned_path_head, "aggregate_vehicles.csv", True)

### Health insurance

##### JPL Notes Cleaning

In [ ]:
health_insurance_df = datasets['health_insurance']
health_insurance_df.head(2)

In [ ]:
cleaned_health_insurance_df = health_insurance_df.iloc[:, :2].copy()

cols_to_sum = []

for col in health_insurance_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        cols_to_sum.append(col)

total_col = ('CALCULATED', 'No Health Insurance Coverage')
cleaned_health_insurance_df[total_col] = health_insurance_df[cols_to_sum].sum(axis=1)

cleaned_health_insurance_df.head(2)

In [ ]:
export_census_csv(cleaned_health_insurance_df, cleaned_path_head, "health_insurance.csv", True)

### Households with a computer

In [ ]:
households_w_computer_df = datasets['households_w_computer']
households_w_computer_df.head(2)

In [ ]:
export_census_csv(households_w_computer_df, cleaned_path_head, "households_w_computer.csv", True)

### Internet subscription

In [ ]:
internet_subscription_df = datasets['internet_subscription']
internet_subscription_df.head(2)

In [ ]:
export_census_csv(internet_subscription_df, cleaned_path_head, "internet_subscription.csv", True)

### Limited English speaking

##### JPL Notes Cleaning

In [ ]:
limited_english_speaking_df = datasets['limited_english_speaking']
limited_english_speaking_df.head(2)

In [ ]:
cols_to_sum = []

for col in limited_english_speaking_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        cols_to_sum.append(col)

cleaned_limited_english_speaking_df = limited_english_speaking_df.copy()
total_col = ('CALCULATED', 'Total Limited English Speaking Households')
cleaned_limited_english_speaking_df.insert(2, total_col, limited_english_speaking_df[cols_to_sum].sum(axis=1))

cleaned_limited_english_speaking_df.head(2)

In [ ]:
export_census_csv(cleaned_limited_english_speaking_df, cleaned_path_head, "limited_english_speaking.csv", True)

### Living Arrangements

##### JPL Notes Cleaning

In [ ]:
living_arrangements_df = datasets['living_arrangements']
living_arrangements_df.head(2)

In [ ]:
cols_to_sum = []
for col in living_arrangements_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        cols_to_sum.append(col)

cleaned_living_arrangements_df = living_arrangements_df.copy()
total_col = ('CALCULATED', r'Total "Living alone"')
cleaned_living_arrangements_df.insert(2, total_col, living_arrangements_df[cols_to_sum].sum(axis=1))
cleaned_living_arrangements_df.head(2)

In [ ]:
export_census_csv(cleaned_living_arrangements_df, cleaned_path_head, "living_arrangements.csv", True)

### Income share of FPL

##### JPL Notes Cleaning

In [ ]:
income_share_of_fpl_df = datasets['income_share_of_fpl']
income_share_of_fpl_df.head(2)

In [ ]:
cleaned_fpl_df = income_share_of_fpl_df.iloc[:, :3].copy()

under_1_cols = []
under_1_5_cols = []
under_2_cols = []

for col in income_share_of_fpl_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!'):
        col_alias = col[1]
        
        # Under 1.0
        if 'Under .50' in col_alias or '.50 to .99' in col_alias:
            under_1_cols.append(col)
            under_1_5_cols.append(col)
            under_2_cols.append(col)
        
        # 1.0 to 1.5
        elif '1.00 to 1.24' in col_alias or '1.25 to 1.49' in col_alias:
            under_1_5_cols.append(col)
            under_2_cols.append(col)
        
        # 1.5 to 2.0
        elif '1.50 to 1.84' in col_alias or '1.85 to 1.99' in col_alias:
            under_2_cols.append(col)

cleaned_fpl_df[('CALCULATED', 'Total Under 100% FPL')] = income_share_of_fpl_df[under_1_cols].sum(axis=1)
cleaned_fpl_df[('CALCULATED', 'Total Under 150% FPL')] = income_share_of_fpl_df[under_1_5_cols].sum(axis=1)
cleaned_fpl_df[('CALCULATED', 'Total Under 200% FPL')] = income_share_of_fpl_df[under_2_cols].sum(axis=1)

cleaned_fpl_df.head(3)

In [ ]:
export_census_csv(cleaned_fpl_df, cleaned_path_head, "income_share_of_fpl.csv", True)

### Persons under 5 & 65

##### JPL Notes Cleaning

In [ ]:
person_under_5_65_df = datasets['person_under_5_65']
person_under_5_65_df.head(2)

In [ ]:
cleaned_genders_df = person_under_5_65_df.iloc[:, :2].copy()
cleaned_genders_df[('B01001_002E', 'Estimate!!Total:!!Male:')] = person_under_5_65_df[('B01001_002E', 'Estimate!!Total:!!Male:')]
cleaned_genders_df[('B01001_026E', 'Estimate!!Total:!!Female:')] = person_under_5_65_df[('B01001_026E', 'Estimate!!Total:!!Female:')]

In [ ]:
export_census_csv(cleaned_genders_df, cleaned_path_head, "genders.csv", True)

In [ ]:
cleaned_males_df = person_under_5_65_df.iloc[:, :2].copy()

males_under_5_cols = []
males_under_18_cols = []
males_over_65_cols = []

for col in person_under_5_65_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!Male:!!'):
        col_alias = col[1]
        
        # Under 5
        if 'Under 5' in col_alias:
            males_under_5_cols.append(col)
            males_under_18_cols.append(col)
        
        # Under 18
        elif ('5 to 9' in col_alias or '10 to 14' in col_alias or 
              '15 to 17' in col_alias):
            males_under_18_cols.append(col)
        
        # 65 and over
        elif ('65 and 66' in col_alias or '67 to 69' in col_alias or
              '70 to 74' in col_alias or '75 to 79' in col_alias or
              '80 to 84' in col_alias or '85' in col_alias):
            males_over_65_cols.append(col)

cleaned_males_df[('CALCULATED', 'Males Under 5')] = person_under_5_65_df[males_under_5_cols].sum(axis=1)
cleaned_males_df[('CALCULATED', 'Males Under 18')] = person_under_5_65_df[males_under_18_cols].sum(axis=1)
cleaned_males_df[('CALCULATED', 'Males Over 65')] = person_under_5_65_df[males_over_65_cols].sum(axis=1)

cleaned_males_df.head(2)

In [ ]:
export_census_csv(cleaned_males_df, cleaned_path_head, "person_under_5_65_males.csv", True)

In [ ]:
cleaned_females_df = person_under_5_65_df.iloc[:, :2].copy()

females_under_5_cols = []
females_under_18_cols = []
females_over_65_cols = []

for col in person_under_5_65_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total:!!Female:!!'):
        col_alias = col[1]
        
        # Under 5
        if 'Under 5' in col_alias:
            females_under_5_cols.append(col)
            females_under_18_cols.append(col)
        
        # Under 18
        elif ('5 to 9' in col_alias or '10 to 14' in col_alias or 
              '15 to 17' in col_alias):
            females_under_18_cols.append(col)
        
        # 65 and over
        elif ('65 and 66' in col_alias or '67 to 69' in col_alias or
              '70 to 74' in col_alias or '75 to 79' in col_alias or
              '80 to 84' in col_alias or '85' in col_alias):
            females_over_65_cols.append(col)

cleaned_females_df[('CALCULATED', 'Females Under 5')] = person_under_5_65_df[females_under_5_cols].sum(axis=1)
cleaned_females_df[('CALCULATED', 'Females Under 18')] = person_under_5_65_df[females_under_18_cols].sum(axis=1)
cleaned_females_df[('CALCULATED', 'Females Over 65')] = person_under_5_65_df[females_over_65_cols].sum(axis=1)

cleaned_females_df.head(2)

In [ ]:
export_census_csv(cleaned_females_df, cleaned_path_head, "person_under_5_65_females.csv", True)

### Population in group quarters

In [ ]:
population_group_quarters_df = datasets['population_group_quarters']
population_group_quarters_df.head(2)

In [ ]:
export_census_csv(population_group_quarters_df, cleaned_path_head, "population_group_quarters.csv", True)

### Race origin

In [ ]:
race_origin_df = datasets['race_origin']
race_origin_df.head(2)

In [ ]:
export_census_csv(race_origin_df, cleaned_path_head, "race_origin.csv", True)

### Tenure

In [ ]:
tenure_df = datasets['tenure']
tenure_df.head(2)

In [ ]:
export_census_csv(tenure_df, cleaned_path_head, "tenure.csv", True)

### 2022 Census Hawaiian Homelands

In [ ]:
tenure_df = datasets['tenure']
tenure_df.head(2)

In [ ]:
# export_census_csv(tenure_df, cleaned_path_head, "2022_census_hawaiian_homelands.csv", True)

##### JPL Notes Cleaning

In [ ]:
hawaiian_homelands_df = datasets['2022_census_hawaiian_homelands']
hawaiian_homelands_df.head(2)

In [ ]:
cleaned_hawaiian_homelands_df = hawaiian_homelands_df.iloc[:, :2].copy()

under_5_cols = []
under_18_cols = []
over_65_cols = []

for col in hawaiian_homelands_df.columns:
    if len(col) == 2 and col[1].startswith('Estimate!!Total!!Total population!!'):
        col_alias = col[1]
        
        # Under 5
        if 'Under 5' in col_alias:
            under_5_cols.append(col)
            under_18_cols.append(col)
        
        # Under 18
        elif '5 to 17' in col_alias:
            under_18_cols.append(col)
        
        # 65 and over
        elif ('65 to 74' in col_alias or '75' in col_alias):
            over_65_cols.append(col)

cleaned_hawaiian_homelands_df[('CALCULATED', 'Total Population Under 5')] = hawaiian_homelands_df[under_5_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
cleaned_hawaiian_homelands_df[('CALCULATED', 'Total Population Under 18')] = hawaiian_homelands_df[under_18_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
cleaned_hawaiian_homelands_df[('CALCULATED', 'Total Population Over 65')] = hawaiian_homelands_df[over_65_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)

cleaned_hawaiian_homelands_df.head(2)

In [ ]:
cleaned_hawaiian_homelands_df = pd.concat([cleaned_hawaiian_homelands_df, hawaiian_homelands_df.iloc[:, 10:-4].copy()], axis=1)
cleaned_hawaiian_homelands_df.head(2)

In [ ]:
export_census_csv(cleaned_hawaiian_homelands_df, cleaned_path_head, "2022_census_hawaiian_homelands.csv", True)

##### Add Hawaiian Homelands census poverty data to FPL poverty status dataset

In [ ]:
cleaned_fpl_df = pd.concat([cleaned_fpl_df, hawaiian_homelands_df.iloc[:, -4:].copy()], axis=1)
cleaned_fpl_df.head(2)

In [ ]:
export_census_csv(cleaned_fpl_df, cleaned_path_head, "income_share_of_fpl.csv", True)

## Add Proportions to All Datasets ====================================

In [ ]:
# Block groups population from 2020 Census
block_groups_geojson_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/block-groups/2020_Census_Block_Groups_Stripped.geojson")
hawaiian_homelands_geojson_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/block-groups/Census_Hawaiian_Homelands_hhl10_Stripped.geojson")

# Load block groups populations
with open(block_groups_geojson_path, 'r') as f:
    block_groups_data = json.load(f)

# Create mapping of geoid20 -> pop20
block_group_populations = {}
for feature in block_groups_data['features']:
    geoid = str(feature['properties']['geoid20'])  # Convert to string for consistent lookup
    pop = feature['properties']['pop20']
    # Store with the full Geography ID format used in CSVs
    block_group_populations[f"1500000US{geoid}"] = pop

# Load Hawaiian Homelands populations
with open(hawaiian_homelands_geojson_path, 'r') as f:
    hawaiian_homelands_data = json.load(f)

# Create mapping of GEOID10 -> POP10
hawaiian_homelands_populations = {}
for feature in hawaiian_homelands_data['features']:
    geoid = str(feature['properties']['GEOID10'])  # Convert to string for consistent lookup
    pop = feature['properties']['POP10']
    # Hawaiian homelands uses just the GEOID without prefix
    hawaiian_homelands_populations[geoid] = pop

# Calculate total population across all blocks
total_population_block_groups = sum(block_group_populations.values())
total_population_hawaiian_homelands = sum(hawaiian_homelands_populations.values())
total_population = total_population_block_groups + total_population_hawaiian_homelands

print(f"Total population (block groups): {total_population_block_groups}")
print(f"Total population (Hawaiian homelands): {total_population_hawaiian_homelands}")
print(f"Total population (combined): {total_population}")

## ============================================================

In [ ]:
# import glob

# csv_files = glob.glob(os.path.join(cleaned_path_head, "*.csv"))

# # Process each CSV file
# for csv_file in csv_files:
#     print(f"Processing {csv_file}...")
#     try:
#         census_csv_to_json(csv_file, json_dir_path)
#     except Exception as e:
#         print(f"Error processing {csv_file}: {e}")

In [ ]:
# Process each cleaned CSV to add proportion columns
def add_proportions_to_csv(csv_path, total_population, block_group_populations, hawaiian_homelands_populations):
    """
    Add block proportion and total proportion columns to a cleaned CSV.
    Also adds the official Census_Population column from GeoJSON.

    Args:
        csv_path: Path to the cleaned CSV file
        total_population: Total population across all blocks
        block_group_populations: Dictionary mapping Geography IDs to populations for block groups
        hawaiian_homelands_populations: Dictionary mapping GEOIDs to populations for Hawaiian homelands
    """
    df = pd.read_csv(csv_path)

    filename = os.path.basename(csv_path)
    is_hawaiian_homelands = 'hawaiian_homelands' in filename.lower()

    # Map each row to its population from GeoJSON
    if is_hawaiian_homelands:
        # For Hawaiian Homelands, extract the GEOID from Geography
        # Geography format has prefix "2500000US" + GEOID (e.g., "2500000US5003")
        # Need to strip the prefix to match the population dictionary keys
        def get_hh_population(geo_id):
            # Convert to string for lookup
            geo_id_str = str(geo_id) if not pd.isna(geo_id) else None
            if geo_id_str is None:
                return None
            
            # Remove the "2500000US" prefix if present
            if 'US' in geo_id_str:
                geo_id_str = geo_id_str.split('US')[1]
            
            # Try direct lookup
            return hawaiian_homelands_populations.get(geo_id_str, None)

        block_populations = df['Geography'].apply(get_hh_population)
        
        # Debug: Check how many populations were found
        found_count = block_populations.notna().sum()
        total_count = len(block_populations)
        print(f"  → Found {found_count}/{total_count} Hawaiian Homelands populations")
        if found_count == 0:
            print(f"  ⚠ WARNING: No populations found! Sample Geography IDs: {df['Geography'].head(3).tolist()}")
            print(f"  ⚠ Available population keys: {list(hawaiian_homelands_populations.keys())[:5]}")
    else:
        # For regular block groups, Geography is already in the right format
        block_populations = df['Geography'].map(block_group_populations)
        
        # Debug: Check how many populations were found
        found_count = block_populations.notna().sum()
        total_count = len(block_populations)
        print(f"  → Found {found_count}/{total_count} block group populations")

    # Add Census_Population column right after Geographic Area Name
    # Check if Census_Population column already exists
    if 'Census_Population' in df.columns:
        # Update existing column
        df['Census_Population'] = block_populations
        print(f"  → Updated existing Census_Population column in {filename}")
    else:
        # Insert new column at position 2 (after Geography and Geographic Area Name)
        insert_position = 2
        df.insert(insert_position, 'Census_Population', block_populations)
        print(f"  → Added Census_Population column to {filename}")

    # Identify metric columns (skip Geography, Geographic Area Name, Census_Population, and Margin of Error columns)
    metric_cols = [col for col in df.columns 
                   if col not in ['Geography', 'Geographic Area Name', 'Census_Population']
                   and not col.startswith('Margin of Error')
                   and '(Per Capita)' not in col]

    # Add proportion columns for each metric
    for col in metric_cols:
        # Skip if already a proportion column or if it's a string column
        if pd.api.types.is_numeric_dtype(df[col]):
            # Add proportion column (metric / block population)
            prop_col = f'{col} (Per Capita)'
            df[prop_col] = df[col] / block_populations

    # Replace inf and NaN with None for proper handling
    df = df.replace([float('inf'), float('-inf')], None)

    return df

In [ ]:
# Process all CSVs in the cleaned directory
csv_files = glob.glob(os.path.join(cleaned_path_head, "*.csv"))

for csv_file in csv_files:
    try:
        print(f"Adding proportions to {os.path.basename(csv_file)}...")
        df_with_props = add_proportions_to_csv(csv_file, total_population, block_group_populations, hawaiian_homelands_populations)
        
        # Save back to the same file
        df_with_props.to_csv(csv_file, index=False)
        print(f"  ✓ Complete")
    except Exception as e:
        print(f"  ✗ Error: {e}")

## Leaflet JSON 

In [ ]:
def clean_column_name(col_name, prefixes_to_remove):
    """
    Remove specified prefixes from column name and clean up remaining !! separators and trailing colons
    """
    # Remove prefixes
    for prefix in prefixes_to_remove:
        if col_name.startswith(prefix):
            col_name = col_name[len(prefix):]
            break

    cleaned_name = col_name.replace('!!', ' ')
    cleaned_name = ' '.join(cleaned_name.split())
    cleaned_name = cleaned_name.rstrip(':')

    return cleaned_name

# This version processes multiple csvs into a single master json with proportion support
def census_csvs_to_master_json(csv_directory, json_path, block_group_populations, hawaiian_homelands_populations):
    """
    Process all census CSV files of a given directory into a single master JSON file.
    Now includes absolute values and proportions (block and total) in nested objects.
    Also includes population data for each geography.
    """       
    csv_directory = os.path.expanduser(csv_directory)
    json_path = os.path.expanduser(json_path)

    os.makedirs(os.path.dirname(json_path), exist_ok=True)

    csv_files = glob.glob(os.path.join(csv_directory, "*.csv"))

    metrics = {}
    prefixes_to_remove = ['Estimate!!Total:!!', 'Estimate!!Total!!', 'Margin of Error!!', '!!Total:!!']

    for csv_file in csv_files:
        csv_filename = os.path.basename(csv_file)
        census_metric_name = os.path.splitext(csv_filename)[0]

        try:
            na_values = ["", "-", "**", "null"]
            df = pd.read_csv(csv_file, na_values=na_values)

            for _, row in df.iterrows():
                geo_id = str(row['Geography']) if not pd.isna(row['Geography']) else None
                if geo_id is None:
                    continue
                    
                original_geo_id = geo_id  # Keep the original for population lookup
                
                # Remove the "1500000US" prefix for the JSON key (only for block groups)
                if isinstance(geo_id, str) and 'US' in geo_id:
                    geo_id = geo_id.split('US')[1]

                geo_name = row['Geographic Area Name']

                if geo_id not in metrics:
                    # Split geographic name
                    geo_name_parts = [geo_name_part.strip() for geo_name_part in geo_name.split(';')]

                    # Determine population based on whether it's Hawaiian Homelands or block group
                    # Handles Hawaiian Homelands data that does not follow typical geo area name structure
                    if len(geo_name_parts) < 4:
                        # Hawaiian Homelands - geo_id is already just the GEOID (e.g., "5271")
                        population = hawaiian_homelands_populations.get(str(geo_id), None)
                        metrics[geo_id] = {
                            "type": "hawaiian_homeland",
                            "name": geo_name,
                            "block_group": None,
                            "census_tract": None,
                            "county": None,
                            "state": None,
                            "population": population,
                            "metrics": {}
                        }
                    else:
                        # Regular block group - original_geo_id has the full format
                        population = block_group_populations.get(original_geo_id, None)
                        metrics[geo_id] = {
                            "type": "block_group",
                            "name": geo_name,
                            "block_group": geo_name_parts[0],
                            "census_tract": geo_name_parts[1],
                            "county": geo_name_parts[2],
                            "state": geo_name_parts[3],
                            "population": population,
                            "metrics": {}
                        }

                # Group metrics by csv
                if census_metric_name not in metrics[geo_id]["metrics"]:
                    metrics[geo_id]["metrics"][census_metric_name] = {}

                # Add all columns (absolute values only, proportions will be calculated after)
                for col in df.columns:
                    if col in ['Geography', 'Geographic Area Name']:
                        continue
                    
                    # Skip Margin of Error columns
                    if col.startswith('Margin of Error'):
                        continue

                    # Clean column name by removing prefixes
                    field_name = clean_column_name(col, prefixes_to_remove)

                    # Initialize nested structure if this is the first time seeing this field
                    if field_name not in metrics[geo_id]["metrics"][census_metric_name]:
                        metrics[geo_id]["metrics"][census_metric_name][field_name] = {
                            "absolute": None,
                            "proportion": None
                        }

                    # Handle the value
                    if pd.isna(row[col]):
                        value = None
                    else:
                        value = int(row[col])

                    # Store the absolute value
                    metrics[geo_id]["metrics"][census_metric_name][field_name]["absolute"] = value

        except Exception as e:
            print(f"Error processing {csv_file}: {e}")
            continue

    # Second pass: Calculate proportions for all metrics
    # proportion = absolute / population for that geography
    
    # Track metrics with proportion > 1.0 for warning
    problematic_metrics = {}
    
    for geo_id in metrics:
        population = metrics[geo_id]["population"]
        
        # Skip if no population data
        if population is None or population == 0:
            continue
            
        for dataset_name in metrics[geo_id]["metrics"]:
            for field_name in metrics[geo_id]["metrics"][dataset_name]:
                absolute_value = metrics[geo_id]["metrics"][dataset_name][field_name]["absolute"]
                
                if absolute_value is not None:
                    proportion = absolute_value / population
                    metrics[geo_id]["metrics"][dataset_name][field_name]["proportion"] = proportion
                    
                    # Track if proportion > 1.0
                    if proportion > 1.0:
                        key = (dataset_name, field_name)
                        if key not in problematic_metrics:
                            problematic_metrics[key] = proportion
                        else:
                            problematic_metrics[key] = max(problematic_metrics[key], proportion)
                else:
                    metrics[geo_id]["metrics"][dataset_name][field_name]["proportion"] = None
    
    # Print warnings for metrics with proportion > 1.0
    if problematic_metrics:
        print("\n⚠ WARNING: The following metrics have proportions > 1.0:")
        print("These may not be appropriate for proportion calculation (e.g., aggregate/total values)\n")
        for (dataset_name, field_name), max_prop in sorted(problematic_metrics.items()):
            print(f"  • {dataset_name} → '{field_name}' (max: {max_prop:.4f})")
        print()

    # Write to JSON file
    with open(json_path, 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f"JSON at {json_path}")
    return metrics

def generate_dataset_params(csv_directory, json_path, block_group_populations, hawaiian_homelands_populations):
    csv_directory = os.path.expanduser(csv_directory)
    json_path = os.path.expanduser(json_path)
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    csv_files = glob.glob(os.path.join(csv_directory, "*.csv"))
    dataset_params = {}
    prefixes_to_remove = ['Estimate!!Total:!!', 'Estimate!!Total!!', 'Margin of Error!!', '!!Total:!!']
    
    # Define color schemes
    color_schemes = {
        'viridis': ['#fde725', '#b5de2b', '#6ece58', '#35b779', '#1f9e89', '#26828e', '#31688e', '#3e4989', '#482878', '#440154'],
        'reds': ['#FFEDA0', '#FED976', '#FEB24C', '#FD8D3C', '#FC4E2A', '#E31A1C', '#BD0026', '#800026', '#5A0018', '#3A000F'],
        'blues': ['#f7fcf0', '#e0f3db', '#ccebc5', '#a8ddb5', '#7bccc4', '#4eb3d3', '#2b8cbe', '#0868ac', '#084081', '#042652']
    }
    
    for csv_file in csv_files:
        csv_filename = os.path.basename(csv_file)
        key = os.path.splitext(csv_filename)[0]
        hawaiian_homelands = "hawaiian_homelands" in csv_filename.lower()
        dataset_params[key] = {
            'metricName': '',
            'metricLabel': '',
            'hawaiianHomelands': hawaiian_homelands,
            'columnThresholds': {},
        }
        na_values = ["", "-", "**", "null"]
        df = pd.read_csv(csv_file, na_values=na_values)
        
        # Calculate proportions for this dataframe
        proportions_df = df.copy()
        for _, row in df.iterrows():
            geo_id = str(row['Geography']) if not pd.isna(row['Geography']) else None
            if geo_id is None:
                continue
            
            # Get population for this geography
            if hawaiian_homelands:
                # For Hawaiian Homelands, geo_id might need cleaning
                clean_geo_id = geo_id.split('US')[1] if 'US' in geo_id else geo_id
                population = hawaiian_homelands_populations.get(clean_geo_id, None)
            else:
                population = block_group_populations.get(geo_id, None)
            
            if population is None or population == 0:
                continue
            
            # Calculate proportions for each numeric column in this row
            for col in df.columns:
                if col in ['Geography', 'Geographic Area Name'] or col.startswith('Margin of Error'):
                    continue
                
                value = row[col]
                if pd.isna(value):
                    proportions_df.at[row.name, f"{col} (Per Capita)"] = None
                else:
                    proportions_df.at[row.name, f"{col} (Per Capita)"] = value / population
        
        # Now generate thresholds using the proportions we just calculated
        for col in df.columns:
            if col in ['Geography', 'Geographic Area Name', 'Census_Population']:
                continue
            
            # Skip Margin of Error columns
            if col.startswith('Margin of Error'):
                continue
            
            # Clean column name by removing prefixes
            field_name = clean_column_name(col, prefixes_to_remove)
            
            # Use the calculated proportion column
            pct_col_name = f"{col} (Proportion)"
            if pct_col_name not in proportions_df.columns:
                print(f"  Warning: No proportion calculated for '{field_name}' in {csv_filename} - skipping")
                continue
            
            numeric_col = pd.to_numeric(proportions_df[pct_col_name], errors='coerce')
            
            # Drop NaN values before calculating quantiles
            numeric_col_clean = numeric_col.dropna()
            
            # Skip if there's not enough data for quantiles
            if len(numeric_col_clean) < 10:
                print(f"  Warning: Skipping '{field_name}' in {csv_filename} - insufficient data ({len(numeric_col_clean)} values)")
                continue
            
            try:
                labels, edges = pd.qcut(numeric_col_clean, q=10, labels=False, retbins=True, duplicates='drop')
                
                # For proportions: round to 3 decimal places (0.001 precision)
                rounded_edges = [round(edge, 3) for edge in edges if not pd.isna(edge)]
                
                threshold_config = {
                    'thresholds': rounded_edges,
                    'colorSchemes': color_schemes,
                }
                
                dataset_params[key]['columnThresholds'][field_name] = threshold_config
                
            except Exception as e:
                print(f"  Warning: Could not create thresholds for '{field_name}' in {csv_filename}: {e}")
    
    with open(json_path, 'w') as f:
        json.dump(dataset_params, f, indent=2)
    
    return dataset_params

In [ ]:
json_output_path = os.path.join(json_dir_path,"census_metrics_by_block_group.json") 

metrics = census_csvs_to_master_json(cleaned_path_head, json_output_path, block_group_populations, hawaiian_homelands_populations)

In [ ]:
# cleaned_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/cleaned-data")

# json_dir_path = os.path.expanduser("~/Desktop")
json_output_path = os.path.join(json_dir_path,"census_datasets_config.json") 

dataset_params = generate_dataset_params(
    cleaned_path_head, 
    json_output_path,
    block_group_populations,
    hawaiian_homelands_populations
)